In [1]:
import pandas as pd
import numpy as np
import scanpy as sc
from utag import utag
import anndata

/Users/yakir/miniconda3/envs/utag/lib/python3.9/site-packages/numba/core/decorators.py:246: RuntimeWarning: nopython is set for njit and is ignored
  warnings.warn('nopython is set for njit and is ignored', RuntimeWarning)
/Users/yakir/miniconda3/envs/utag/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def run_utag(dsetname):
    os.makedirs(f'_embeddings', exist_ok=True)

    d = sc.read_h5ad(f'_data/{dsetname}/cells.h5ad')
    adata = anndata.AnnData(
        X = np.array(d.X).astype(np.float64),
        obs = d.obs[['sid']],
        var = d.var
    )
    adata.obsm['spatial'] = np.array(d.obsm['spatial'])

    utag_results = utag(
        adata,
        slide_key="sid",
        max_dist=15,
        normalization_mode='l1_norm',
        apply_clustering=True,
        clustering_method = 'leiden', 
        resolutions = [0.3]
    )
    sc.tl.umap(utag_results)
    utag_results.obs.rename(columns={
        'UTAG Label_leiden_0.3':'leiden_0.3',
        'UTAG Label_leiden_0.1':'leiden_0.1',
    }, inplace=True)
    utag_results.write(f'_embeddings/{dsetname}_utag_noharm.h5ad')

# Run

In [ ]:
run_utag('ALZ')

In [ ]:
run_utag('UC')